# Solution Comparison Dashboard

Compare two `output_payload` JSON solutions side-by-side.

- **Solution Overview** — key metrics table with delta columns
- **Gantt charts** — driver timelines for both solutions on a shared Y axis
- **Distance per service** — side-by-side stacked bars
- **Distance per driver** — side-by-side stacked bars
- **Summary metrics** — KPI comparison with delta columns (requires eval reports)

All logic lives in `src/analysis/compare_solutions.py`. Edit the **Configuration** cell only.

> **Baseline note**: solutions without `addData` (historic snapshots) have `duration_min=0`;
> their VEHICLE_TRANSPORTATION bars are rendered with a 5-minute minimum width as position markers.

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
from typing import Optional


from alfred.analysis.compare_solutions import (
    load_and_prepare,
    build_overview_table,
    build_comparison_gantt_figure,
    build_comparison_service_figure,
    build_comparison_driver_figure,
    build_comparison_summary_table,
    build_comparison_route_map,
)
from alfred.optimization.settings.model_params import ModelParams
from alfred.optimization.settings.solver_settings import DEFAULT_DISTANCE_METHOD

_PROJECT_ROOT = Path("../..").resolve()


In [4]:
# ── Configuration ──────────────────────────────────────────────────────────────────────
# Uncomment ONE experiment block below, then run the notebook.

# ── Phase 2 · S1-Speed (OSRM distance + speed-based times) ────────────────────────
# _SOL_A       = _PROJECT_ROOT / "experiments/phase2/cases/file_snapshots/output_payload_scoped.json"
# _SOL_B       = _PROJECT_ROOT / "experiments/phase2/results/setup_1/output_speed/output_payload.json"
# _INPUT_FILE  = _PROJECT_ROOT / "experiments/phase2/cases/input_mirror.json"
# _DRIVER_DIR  = _PROJECT_ROOT / "data/examples/driver_directory.json"
# _EVAL_A      = None
# _EVAL_B      = _PROJECT_ROOT / "experiments/phase2/results/setup_1/output_speed/run-local-d41c6d9d/diagnostics/solution_evaluation_report.json"
# _LABEL_A     = "baseline"
# _LABEL_B     = "s1-speed"
# _DATE        = "2026-02-18"
# _TIME_METHOD = "speed_based"

# ── Phase 2 · S1-Time (OSRM distance + OSRM duration times) ───────────────────────
# _SOL_A       = _PROJECT_ROOT / "experiments/phase2/cases/file_snapshots/output_payload_scoped.json"
# _SOL_B       = _PROJECT_ROOT / "experiments/phase2/results/setup_1/output_time/output_payload.json"
# _INPUT_FILE  = _PROJECT_ROOT / "experiments/phase2/cases/input_mirror.json"
# _DRIVER_DIR  = _PROJECT_ROOT / "data/examples/driver_directory.json"
# _EVAL_A      = None
# _EVAL_B      = None
# _LABEL_A     = "baseline"
# _LABEL_B     = "s1-time"
# _DATE        = "2026-02-18"
# _TIME_METHOD = "osrm_times"

# ── Phase 4 (OFFLINE · 2026-03-13 Bogota) ─────────────────────────────────────────
_SOL_A       = _PROJECT_ROOT / "misc" / "experiments" / "phase8" / "offline" / 'results' /"output" / "output_payload.json"
_SOL_B       = _PROJECT_ROOT / "misc" / "experiments" / "phase8" / "react" / 'results' /"output" / "output_payload.json"
_INPUT_FILE  = _PROJECT_ROOT / "misc" / "experiments" / "phase8" / "api_snapshots" / "pre" / "optimization_input_snapshot.json"
_DRIVER_DIR  = _PROJECT_ROOT / "misc" / "experiments" / "phase8" / "api_snapshots" / "pre" / "driver_directory_snapshot.json"
_EVAL_A      = None
_EVAL_B      = None
_LABEL_A     = "offline"
_LABEL_B     = "react"
_DATE        = "2026-05-19"
_TIME_METHOD = "osrm_times"

# ── Active ────────────────────────────────────────────────────────────────────────────
SOL_A_PAYLOAD:    Optional[Path] = _SOL_A
SOL_B_PAYLOAD:    Optional[Path] = _SOL_B
INPUT_FILE:       Optional[Path] = _INPUT_FILE
DRIVER_DIRECTORY: Optional[Path] = _DRIVER_DIR
EVAL_A:           Optional[Path] = _EVAL_A
EVAL_B:           Optional[Path] = _EVAL_B
LABEL_A:          str            = _LABEL_A
LABEL_B:          str            = _LABEL_B
PLANNING_DATE:    Optional[str]  = _DATE
DISTANCE_METHOD:  str            = DEFAULT_DISTANCE_METHOD
TIME_METHOD:      str            = _TIME_METHOD

_params = ModelParams()
ALFRED_SPEED_KMH: float = _params.alfred_speed_kmh
print(f"alfred_speed_kmh={ALFRED_SPEED_KMH} | distance_method={DISTANCE_METHOD!r} | time_method={TIME_METHOD!r}")

alfred_speed_kmh=20.0 | distance_method='osrm' | time_method='osrm_times'


In [5]:
# ── Load and prepare both solutions ──────────────────────────────────────────────────
pair = load_and_prepare(
    sol_a=SOL_A_PAYLOAD,
    sol_b=SOL_B_PAYLOAD,
    input_file=INPUT_FILE,
    driver_directory=DRIVER_DIRECTORY,
    planning_date=PLANNING_DATE,
    distance_method=DISTANCE_METHOD,
    speed_kmh=ALFRED_SPEED_KMH,
)

[driver_home_lookup] 14 drivers loaded
[coord_lookup] 36 labors | method='osrm'
sol_a: 34 labors | 13 drivers | 79 segments
sol_b: 30 labors | 13 drivers | 71 segments


---
## Solution Overview

In [6]:
build_overview_table(pair, LABEL_A, LABEL_B)

,offline,react,delta,delta_pct
metric,,,,
services,30.00,28.00,-2.00,-6.67
labors_total,34.00,30.00,-4.00,-11.76
labors_vt,32.00,30.00,-2.00,-6.25
labors_non_vt,2.00,0.00,-2.00,-100.00
drivers_used,13.00,13.00,0.00,0.00
labors_assigned,32.00,30.00,-2.00,-6.25
labors_infeasible,0.00,0.00,0.00,NaN
labors_infeasible_pct,0.00,0.00,0.00,NaN
labors_in_grace,0.00,0.00,0.00,NaN


---
## Gantt Charts — Driver Timelines

In [7]:
build_comparison_gantt_figure(pair, LABEL_A, LABEL_B).show()

---
## Distance per Service

In [8]:
build_comparison_service_figure(pair, LABEL_A, LABEL_B).show()

---
## Distance per Driver

In [9]:
build_comparison_driver_figure(pair, LABEL_A, LABEL_B).show()

---
## Route Comparison Map

Select a driver from the dropdown to view both solutions' routes side by side.

Each labor is assigned a distinct color pair — thick line = service leg, thin line = move leg.
Baseline uses the palette forward (labor 1 = color 0, 2 = color 1, …); algorithm uses it in reverse for maximum contrast.
Numbered circles mark labor start points; open rings mark intermediate ends; **F** marks the final end point.

In [10]:
import ipywidgets as widgets
from IPython.display import display
from alfred.analysis.solution_evaluation import build_route_map

# Pixel dimensions for each map — adjust to taste.
_MAP_W, _MAP_H = 640, 520

_drv_dropdown = widgets.Dropdown(
    options=pair.all_drivers,
    description="Driver:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="300px"),
)

_out_a = widgets.Output(layout=widgets.Layout(width=f"{_MAP_W}px"))
_out_b = widgets.Output(layout=widgets.Layout(width=f"{_MAP_W}px"))

_map_row = widgets.HBox(
    [
        widgets.VBox(
            [widgets.HTML(f"<h4 style='text-align:center;margin:4px 0'>{LABEL_A}</h4>"), _out_a],
        ),
        widgets.VBox(
            [widgets.HTML(f"<h4 style='text-align:center;margin:4px 0'>{LABEL_B}</h4>"), _out_b],
        ),
    ],
)


def _render_maps(driver_id):
    home_wkt = (pair.driver_home_lookup or {}).get(str(driver_id))

    _out_a.clear_output(wait=True)
    with _out_a:
        try:
            display(build_route_map(
                services=None,
                rows=pair.rows_a,
                driver_id=driver_id,
                driver_home_wkt=home_wkt,
                label=LABEL_A,
                points_lookup=pair.points_lookup,
                reverse_palette=False,
                map_width=_MAP_W,
                map_height=_MAP_H,
            ))
        except Exception as e:
            print(f"{e}")

    _out_b.clear_output(wait=True)
    with _out_b:
        try:
            display(build_route_map(
                services=None,
                rows=pair.rows_b,
                driver_id=driver_id,
                driver_home_wkt=home_wkt,
                label=LABEL_B,
                points_lookup=pair.points_lookup,
                reverse_palette=True,
                map_width=_MAP_W,
                map_height=_MAP_H,
            ))
        except Exception as e:
            print(f"{e}")


_drv_dropdown.observe(lambda change: _render_maps(change["new"]), names="value")
_render_maps(_drv_dropdown.value)
display(_drv_dropdown, _map_row)


Dropdown(description='Driver:', layout=Layout(width='300px'), options=('10451', '10500', '11988', '119889', '1…

---
## Summary Metrics
_Requires `EVAL_A` / `EVAL_B` to be set._

In [9]:
tbl = build_comparison_summary_table(EVAL_A, EVAL_B, LABEL_A, LABEL_B)
if tbl is not None:
    display(tbl)

No evaluation reports provided — set EVAL_A / EVAL_B to enable this section.
